In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import sys
import metpy
import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import metpy.calc as mpcalc
import pandas as pd
from netCDF4 import Dataset
import os
import glob
from datetime import datetime
import seaborn as sns
import netCDF4
from netCDF4 import Dataset
from metpy.units import units
import dask
import xarray as xr
from shapely import Polygon
import regionmask
import geopandas as gpd
import dask
from scipy.stats import circmean

In [2]:
from dask.distributed import Client
client = Client(threads_per_worker=1,memory_limit=None,n_workers=28)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 28
Total threads: 28,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41129,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:37279,Total threads: 1
Dashboard: /proxy/35497/status,Memory: 0 B
Nanny: tcp://127.0.0.1:39721,


In [3]:
def get_file_paths(variable):
    list = ["01.nc","02.nc","03.nc","04.nc"]
    list.sort()
    file_paths = []
    for i in list:
        fp = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/"+variable+"/latest/"
        all_files = [os.path.join(root, f) for root, _, files in os.walk(fp)
            for f in files
            if f.startswith(variable+"_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_1hr_") and f.endswith(i)]
        file_paths.extend(all_files)
    file_paths.sort()
    return file_paths 

In [7]:
def calc_mean_towns(file_path,variable,unit: str = "m/s"):
    with xr.open_mfdataset(file_path, engine="h5netcdf", chunks="auto") as ds:
        towns = ds[variable].sel(lat=slice(-20.768799,-18.0708),lon=slice(145.12054,147.9812)).mean(dim=["lat","lon"])
        return towns*units(unit)

In [6]:
def calc_mean_cairns(file_path,variable,unit: str = "m/s"):
    with xr.open_mfdataset(file_path, engine="h5netcdf", chunks="auto") as ds:
        cairns = ds[variable].sel(lat=slice(-18.165955,-15.468018),lon=slice(144.27374,147.09222)).mean(dim=["lat","lon"])                  
        return cairns*units(unit)

In [7]:
# function for calculating mean 850hPa wind regimes over the radar domains
def calc_mean_willis(file_path,variable,unit: str = "m/s"):
    with xr.open_mfdataset(file_path, engine="h5netcdf", chunks="auto") as ds:               
        willis = ds[variable].sel(lat=slice(-17.636353,-14.938416),lon=slice(148.55927,151.36993)).mean(dim=["lat","lon"])
        return willis*units(unit)

# Wind regime dataset creation from BARRA-R2 850 hPa winds

In [4]:
def create_wind_regime_ds(wind_speed: xr.DataArray,wind_dir: xr.DataArray, region: str):
    time_values = wind_dir.time.values
    wind_speed_array = xr.DataArray(wind_speed.values, dims=('time'), coords={'time': time_values})
    wind_dir_array = xr.DataArray(wind_dir.values,dims=('time'), coords={'time': time_values})
    wind_ds = xr.Dataset({'wind_dir':wind_dir_array,
                         'wind_speed':wind_speed_array}, 
            attrs={'note':f'JFMA 850 hPa winds for the {region} radar domain, created with xarray'})
    return wind_ds

In [16]:
def save_regime_ds_to_netcdf(ds,region: str):
    if 'time' not in ds.dims:
        raise ValueError("Dataset does not have a 'time' dimension.")
        
    time_len = ds.sizes['time']

    encoding = {
        var: {
            'shuffle': True,
            'chunksizes': [time_len],
            'zlib': True,
            'complevel': 5
        } for var in ['wind_dir', 'wind_speed']
    }

    ds.to_netcdf(
        f'barra-2_850hPa-winds_{region}_12LST.nc',
        format='NETCDF4',
        encoding=encoding
    )

In [6]:
ua850 = get_file_paths("ua850")
va850 = get_file_paths("va850")

In [14]:
# u winds
uuT = calc_mean_towns(ua850,"ua850","m/s").compute()
uuC = calc_mean_cairns(ua850,"ua850","m/s").compute()
uuW = calc_mean_willis(ua850,"ua850","m/s").compute()
# v winds
vvT = calc_mean_towns(va850,"va850","m/s").compute()
vvC = calc_mean_cairns(va850,"va850","m/s").compute()
vvW = calc_mean_willis(va850,"va850","m/s").compute()

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 113.74 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
2026-04-21 14:42:48,721 - distributed.protocol.pickle - ERROR - Failed to serialize <class 'pint.Quantity'>.
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/protocol/pickle.py", line 63, in dumps
    result = pickle.dumps(x, **dump_kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
_pickle.PicklingError: Can't pickle <class 'pint.Quantity'>: it's not the same object as pint.Quantity

During handling of the above exception, another exception occurred:



In [15]:
uuT

Magnitude,[-16.341471354166668 -17.42064177684295 -20.925743689903847 ... -1.3284880809294872 -0.8283378405448718 -5.472293169070513]
Units,meter/second


In [ ]:
# wind speeds and calculations
towns_wind_speed = metpy.calc.wind_speed(uuT,vvT).drop_vars(['pressure','crs'])
cairns_wind_speed = metpy.calc.wind_speed(uuC,vvC).drop_vars(['pressure','crs'])
willis_wind_speed = metpy.calc.wind_speed(uuW,vvW).drop_vars(['pressure','crs'])

In [ ]:
towns_wind_dir = metpy.calc.wind_direction(uuT,vvT,convention='from').drop_vars(['pressure','crs'])
cairns_wind_dir = metpy.calc.wind_direction(uuC,vvC,convention='from').drop_vars(['pressure','crs'])
willis_wind_dir = metpy.calc.wind_direction(uuW,vvW,convention='from').drop_vars(['pressure','crs'])

In [ ]:
towns_wind = create_wind_regime_ds(towns_wind_speed,towns_wind_dir,"townsville")
cairns_wind = create_wind_regime_ds(cairns_wind_speed,cairns_wind_dir,"cairns")
willis_wind = create_wind_regime_ds(willis_wind_speed,willis_wind_dir,"willis")

In [ ]:
# save_regime_ds_to_netcdf(towns_wind,"towns")

# Get variables for eqpt calculations

In [7]:
# create dictionary of file paths for pressure levels and variables: temperature and specific humidity
file_path_dict = {}
pressure = [950, 925, 850, 700, 600, 500, 400, 300, 200]

for p in pressure:
    file_path_dict[p] = {
        "temp": get_file_paths(f'ta{p}'),
        "spc_humidity": get_file_paths(f'hus{p}')
    }

In [8]:
def get_mean_over_domain(file_path_dict: dict, calc_mean_function: object):
    results = {}
    pressure = [950, 925, 850, 700, 600, 500, 400, 300, 200]

    for p in pressure:
        # calculate mean temperature and specific humidity at each pressure level over the radar domain
        temp = calc_mean_function(file_path_dict[p]["temp"], f'ta{p}', "K").compute()
        spc_humidity = calc_mean_function(file_path_dict[p]["spc_humidity"], f'hus{p}', "kg/kg").compute()
        
        # given the temperature, specific humidity and pressure, calculate Td and eqpt values for each pressure level
        Td = mpcalc.dewpoint_from_specific_humidity(units.hPa * p, units('kg/kg') * spc_humidity.values)
        eqpt = mpcalc.equivalent_potential_temperature(units.hPa * p, units('K') * temp.values, Td.magnitude * units('degC'))
        
        results[p] = {
            "temp": temp,
            "spc_humidity": spc_humidity,
            "Td": Td,
            "eqpt":eqpt
        }

    return results

In [44]:
def create_eqpt_ds(data_dict: dict, region: str):
    pressure_levels = [950, 925, 850, 700, 600, 500, 400, 300, 200]
    temp_list = []
    spc_humidity_list = []
    Td_list = []
    eqpt_list = []

    for p in pressure_levels:
        temp_list.append(data_dict[p]["temp"].expand_dims(pressure=[p]))
        spc_humidity_list.append(data_dict[p]["spc_humidity"].expand_dims(pressure=[p]))

        time = data_dict[p]["temp"].time
        Td_array = xr.DataArray(data_dict[p]["Td"],dims='time',coords={'time':time})
        eqpt_array = xr.DataArray(data_dict[p]["eqpt"],dims='time',coords={'time':time})
        Td_list.append(Td_array.expand_dims(pressure=[p]))
        eqpt_list.append(eqpt_array.expand_dims(pressure=[p]))

    # Concatenate along the new pressure dimension
    temp = xr.concat(temp_list, dim="pressure")
    spc_humidity = xr.concat(spc_humidity_list, dim="pressure")
    Td = xr.concat(Td_list, dim="pressure")
    eqpt = xr.concat(eqpt_list, dim="pressure")
    
    ds = xr.Dataset({
        "temp": temp,
        "spc_humidity": spc_humidity,
        "Td": Td,
        "eqpt":eqpt,
    }, attrs={
        "note": f"JFMA 1979-current mean temperature, specific humidity, dewpoint temperature and equivalent potential temperature for the {region} radar domain over 1979 to 2024-04, created with xarray"
    })
    ds = ds.drop_vars("crs")
    
    return ds

In [45]:
def save_eqpt_ds_to_netcdf(ds,region: str):
    if 'time' not in ds.dims:
        raise ValueError("Dataset does not have a 'time' dimension.")
    if 'pressure' not in ds.dims:
        raise ValueError("Dataset does not have a 'pressure' dimension.")
        
    time_len = ds.sizes['time']
    pressure_len = ds.sizes['pressure']

    encoding = {
        var: {
            'shuffle': True,
            'chunksizes': [pressure_len,time_len],
            'zlib': True,
            'complevel': 5
        } for var in ['temp', 'spc_humidity','Td','eqpt']
    }

    ds.to_netcdf(
        f'barra-2_eqpt-ds_{region}.nc',
        format='NETCDF4',
        encoding=encoding
    )

In [9]:
# townsville
towns_variables_for_eqpt_calc = get_mean_over_domain(file_path_dict,calc_mean_towns)
ds = create_eqpt_ds(towns_variables_for_eqpt_calc, "Townsville")
save_eqpt_ds_to_netcdf(ds,"towns")

In [48]:
# cairns
cairns_eqpt_calc = get_mean_over_domain(file_path_dict,calc_mean_cairns)
dsC = create_eqpt_ds(cairns_eqpt_calc, "Cairns")
save_eqpt_ds_to_netcdf(dsC,"cairns")

In [50]:
# willis
willis_eqpt_calc = get_mean_over_domain(file_path_dict,calc_mean_willis)
dsW = create_eqpt_ds(willis_eqpt_calc, "Willis Island")
save_eqpt_ds_to_netcdf(dsW,"willis")